## Extract data from SI

In [ ]:
import pymupdf
import os
import re
import pandas as pd

def rfind_pattern_from_middle(text, pattern, loc):
    # Extract substring from start to middle position
    substring = text[:loc]
    # Find pattern matches in the substring
    matches = re.findall(pattern, substring)
    # Get the last match if exists
    matche = matches[-1] if matches else None
    if matche == None:
        return loc - 6
    else:
        return rfind_from_middle(text, matche, loc)

def get_detail(text, pattern, loc):
    # Split text into left and right parts at middle position
    substring = text[:loc]
    latestring = text[loc:]
    # Find pattern matches in both parts
    matches = re.findall(pattern, substring)
    matches2 = re.findall(pattern, latestring)
    # Get last match in left part and first match in right part
    matche = matches[-1] if matches else None
    matche2 = matches2[0] if re.findall(pattern, latestring) else None
    # Calculate start and end positions
    if matche == None:
        b = loc - 6
    else:
        b = rfind_from_middle_help(text, matche, loc)
    if matche2 == None:
        d = -2
    else:
        d = loc + text.find(matche2, loc)
    return b, text[b:d+1]

def find_pattern_from_middle(text, pattern, loc):
    # Extract substring from middle position to end
    substring = text[loc:]
    # Find first pattern match in the substring
    matches = re.findall(pattern, substring)
    matche = matches[0]
    return text.find(matche)

def rfind_from_middle_help(s, target, loc):
    # Find last occurrence of target in left part of string
    left_part = s[:loc]
    index_in_left = left_part.rfind(target)
    return index_in_left


In [ ]:
def rfind_from_middle(s, target, loc):
    left_part = s[:loc]
    index_in_left = left_part.rfind(target)
    if loc - index_in_left < 7:  # If match is too close to middle point
        left_part = s[:index_in_left - 1] 
        index_in_left = left_part.rfind(target)
    if index_in_left != -1:
        # Return position if target found
        return index_in_left
    else:
        return -1

path = "D:\\Jupyter\\SI\\羧基\\"  # Directory containing PDF files
filename = "ol4c03151_si_001" + ".pdf"
document = pymupdf.open(path + filename)  # Open PDF document
print(document, len(document))


Document('D:\Jupyter\SI\羧基\ol4c03151_si_001.pdf') 141


In [ ]:
# Define page ranges to process
ra1 = []
ra2 = range(36, 49)  # Subtract 1 from end?
ra3 = []
ra4 = []
# Combine all ranges or use full document if no ranges specified
page = list(ra1) + list(ra2) + list(ra3) + list(ra4) if (ra1 or ra2 or ra3 or ra4) else range(len(document))

result = "" 
for i in page:  # Process specified page range (Python ranges are start-inclusive, end-exclusive)
    text = document[i].get_text()
    result += text + " "  # Concatenate pages with space separator
    # Remove page number markers
    result = result.replace('S' + str(i + 1), '').replace('S' + str(i), '') \
                  .replace('S-' + str(i + 1), '').replace('S-' + str(i), '')
    
result = result.strip()  # Remove leading/trailing whitespace
result = result.replace('\xa0', '')  # Remove non-breaking spaces
result = result.replace('‐', '-')  # Normalize hyphen characters
print(result)



Daicel Chiralpak AD-H column (10% isopropanol in hexanes, 0.8 mL/min, 254 nm,
minor tr = 18.282 min, major tr = 15.530 min). [α]D25= – 10.14 (c = 0.444, CHCl3). 1H
NMR (400 MHz, CDCl3) δ: 9.16 (d, J = 2.3 Hz, 1H), 8.16 (dd, J = 8.1, 2.2 Hz, 1H),
7.86 (d, J = 8.5 Hz, 2H), 7.49 (d, J = 8.3 Hz, 2H), 7.35 – 7.23 (m, 2H), 4.38 (q, J =
7.1 Hz, 2H), 4.32 (dd, J = 7.7, 5.5 Hz, 1H), 2.55 (s, 3H), 2.50 (dd, J = 14.1, 7.7 Hz,
1H), 2.01 (dd, J = 14.1, 5.5 Hz, 1H), 1.38 (t, J = 7.1 Hz, 3H), 0.83 (s, 9H) ppm. 13C
NMR (101 MHz, CDCl3) δ: 197.7, 168.2, 150.7, 149.7, 140.9, 137.6, 136.0, 128.7,
128.1, 125.4, 122.5, 61.2, 50.7, 48.4, 31.5, 30.0, 26.5, 14.2 ppm. FTIR (neat, cm–1) ν:
3819, 3742, 2960, 1722, 1593, 1471, 1366, 1263, 1106, 1025, 800, 597. HRMS (ESI)
m/z: [M + H]+ calcd for C22H28NO3 354.2069; Found 354.2070.
(R)-1-(4-(3-methyl-1-(pyridin-2-yl)butyl)phenyl)ethan-1-one (5d)
The reaction was performed following the General Procedure
with NiBr2∙glyme (7.0 mg, 20 mol%), L1 (10.2 mg, 0.022 mmol),

In [ ]:
def retrieve_other_data(text,item_,k,dot,IR1):
#     data_str=['IR','HNMR','13C{1H}','19F{1H}','HRMS','LRMS','GC-MS','HPLC','Rf','HRES','TLC','found','MS\(EI','alcd','MP']
    # data_str=['IR','H-NMR','C-NMR','F-NMR','HRMS','LR-MS','GC-MS','HPLC','Rf','HRES','TLC','found','MS\(','Calcd','MP']  
    data_str=['IR','HNMR','CNMR','FNMR','HRMS','LRMS','GC-MS','HPLC','Rf','HRES','TLC','ound','MS\(','alcd','MP']       #
    var_name = data_str[k]
    text = text.replace(' ', '')
    text = text.replace('\n', '')
    
    if k==11:
        dot='.'
    match = None
    if dot != '.':
        pattern = f'{var_name}([^{dot}]*)'
        match = re.search(pattern, text)
    else:
        start = text.find(var_name)
        if start!=-1:
            for i in range(len(text)):
                if i>start and text[i] == '.':
                    if not (i > 0 and text[i-1].isdigit() and i < len(text) - 1 and text[i+1].isdigit()):
                        if  i+2<len(text) and text[i+2]=='o':
                            continue
                        else:
                            match = text[start:i+1]
                        break
    if match:
        str_m = match.group() if not isinstance(match, str) else match
        if var_name !=data_str[0]:                                                  #
            item_.append(str_m)
        else:
            if str_m is not None:
                IR1.append(str_m)
            else:
                IR1.append(' ')
            if '(C=O' in str_m:
                words_new = re.findall(r'(\d{4}(?:\.\d+)?)\(C=O', str_m)
            elif '(s)' in str_m:
                words_new = re.findall(r'(\d{4}(?:\.\d+)?)\(s', str_m)
            else:
                str_m=str_m.replace('cm',' ').replace('v',' ').replace('s',' ').replace('m',' ').replace('γ',' ').replace(',',' ')
                IR_value=re.findall(r'\b(\d{4}(?:\.\d+)?)\b',str_m)
                words_new = [s for s in IR_value if re.fullmatch(r'\d+(\.\d+)?', s)]   
            filtered_numbers = [str(i) for i in words_new if 1600. <= float(i) <= 1800.]
            j = ' '.join(filtered_numbers)
            item_.append(j)
    else:
        item_.append(' ')  # returen None


In [ ]:
start=0
name,IR,HNMR,CNMR,FNMR,HRMS,LRMS,GC_MS,HPLC,Rf,IR1,HRES,TLC,c3,found,MS,Anal_Calcd,MP=[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[]
data = [IR,HNMR,CNMR,FNMR,HRMS,LRMS,GC_MS,HPLC,Rf,HRES,TLC,found,MS,Anal_Calcd,MP]
IR_num=result.count("IR")
print(IR_num)
for num in range(IR_num):
    c=result.find("IR",start)        #定位词组、光谱   "IR ("
    
#     b=rfind_from_middle(result,"general",c)    #名字————定位后方 理论b     二次定位词   (\d+.To a)|(\d+.Phos)
#     b1=rfind_pattern_from_middle(result,"\(\d+[']?[a-zA-Z]{0,3}\)",c) #定化合物编号 b1  \([S]?\d+[a-z]{0,3}['′’″\"]?[\),-]
    b1,detail = get_detail(result,"The reaction",c)                       #
    name_a = "\n"                                                   # '.'  "\n"
    print(result[b1-4:b1+7])
    c3.append(result[b1-5:b1+6])
    a= rfind_from_middle(result,name_a,b1-5)   #名字————定位前方 理论a  rfind_from_middle     #
#     a=rfind_pattern_from_middle(result,"\([a-zA-Z]+\)",b1)
    if result[a+1]=="["and name_a=='.':                   #取名字     \([a-zA-Z]{0,2}\d+['′\"]?[^%]\)
        name.append(result[a+4:b1])
    else :
        name.append(result[a+1:b1])                                                          #  

    for k in range(15):
        retrieve_other_data( detail, data[k] , k , '.',IR1)                                  #
    start=c+9
    
for p in range(len(name)):                       #去掉不需要的字符
    c3[p]=c3[p].replace('\n','') if '\n' in c3[p] else c3[p]
    name[p]=name[p].lstrip().rstrip()
    name[p]=name[p].replace('\n','').replace('Preparation of','').replace('Synthesis of','')
    te=name[p]
#     A=te.find(':')
#     name[p]=te[A+1:]
    name[p]=name[p].lstrip().rstrip()
#     te=name[p]
#     A=te.find(' ')
    # name[p]=te[A:]
    # B=te.rfind(' ')
    # name[p]=te[:B]
print(name)
print(IR)


19
.2 ppm. FTI
5d)
The rea
5e)
The rea
5f)
The rea
5h)
The rea
5h)
The rea
5j)
The rea
5k)
The rea
5l)
The rea
5m)
The rea
5n)
The rea
5o)
The rea
5p)
The rea
5q)
The rea
5r)
The rea
5s)
The rea
5t)
The rea
5u)
The rea
5u)
The rea
['128.1, 125.4, 122.5, 61.2, 50.7, 48.4, 31.5, 30.0, 26.5, 14.2', '(R)-1-(4-(3-methyl-1-(pyridin-2-yl)butyl)phenyl)ethan-1-one', '(R)-1-(4-(2-cyclopentyl-1-(pyridin-2-yl)ethyl)phenyl)ethan-1-one', '(R)-1-(4-(2-cyclohexyl-1-(pyridin-2-yl)ethyl)phenyl)ethan-1-one', '(R)-1-(4-(2-cycloheptyl-1-(pyridin-2-yl)ethyl)phenyl)ethan-1-one', '(R)-1-(4-(2-cycloheptyl-1-(pyridin-2-yl)ethyl)phenyl)ethan-1-one', '(R)-1-(4-(3,3-dimethyl-1-(4-(trifluoromethyl)phenyl)butyl)phenyl)ethan-1-one(5j', '(R)-1-(4-(1-(4-fluorophenyl)-3,3-dimethylbutyl)phenyl)ethan-1-one', '(R)-1-(4-(1-(3-fluorophenyl)-3,3-dimethylbutyl)phenyl)ethan-1-one', '(R)-1-(4-(1-(3-chlorophenyl)-3,3-dimethylbutyl)phenyl)ethan-1-one', '(R)-1-(4-(1-(4-chlorophenyl)-3,3-dimethylbutyl)phenyl)ethan-1-one', '(R)-1-(4-

In [ ]:
print('https://pubs.acs.org/doi/10.1021/acs.orglett.'+filename[2:9])


https://pubs.acs.org/doi/10.1021/acs.orglett.4c03151


In [ ]:
# DOI=['10.1021/jo'+filename[2:-11]]*len(name)
# Publication=['J. Org. Chem.']*len(name)
DOI=['10.1021/acs.orglett.'+filename[2:-11]]*len(name)   # acs.orglett.
Publication=['Org. Lett.']*len(name)

Author=['Jianyou Mao']*len(name)
Year=['2024']*len(name)
data_={'IR':IR,'DOI':DOI,'Corresponding Author':Author,'c3':c3,'Publication':Publication,'Year':Year,'name':name,'IR1':IR1,'1HNMR':HNMR,'13CNMR':CNMR,'19FNMR':FNMR,'HRMS':HRMS,'LRMS':LRMS,'GC-MS':GC_MS,'HPLC':HPLC,'Rf':Rf,'HRES':HRES,'TLC':TLC,'found':found,'MS':MS,'Anal_Calcd':Anal_Calcd,'MP':MP}
dataframe = pd.DataFrame(data_,index=range(1,len(name)+1), columns=[ 'smiles','IR','DOI'
                                                                   ,'Corresponding Author','c2','c3','Publication'
                                                                   ,'Year','name','c4','c5','IR1','1HNMR','13CNMR','19FNMR','HRMS','LRMS','GC-MS','HPLC','Rf','HRES','TLC','found','MS','Anal_Calcd','MP'])
path = "D:\\Jupyter\\SI\\12.11\\"  # file path
filename1=filename[:-4]
dataframe.to_excel(path+filename1+".xlsx", index=False)
# dataframe.to_csv(path+filename1+".csv", index=False)


## Name to smiles 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains

import requests
import time
import random
import re
import xlrd
import pandas as pd


In [ ]:
from rdkit import Chem
smiles = 'COc1cc2c3c(=O)c3c1OC(C)(C)OC2(Br)Br'       #
mol = Chem.MolFromSmiles(smiles)  # 将 SMILES 转换为分子对象
if mol:
    canon_smiles = Chem.MolToSmiles(mol, canonical=True)
    print(canon_smiles)
    print(Chem.MolToSmiles(mol, kekuleSmiles=True))
    print(Chem.MolToSmiles(mol, kekuleSmiles=False))


COc1cc2c3c(=O)c3c1OC(C)(C)OC2(Br)Br
COC1=CC2=C3C(=O)C3=C1OC(C)(C)OC2(Br)Br
COc1cc2c3c(=O)c3c1OC(C)(C)OC2(Br)Br


In [ ]:
# ChromeDriver path
driver_path = r'D:\chromedriver-win64\137.0.7151.69\chromedriver.exe'

# set Chrome 
options = Options()
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36")
# options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/88.0.4324.150 Safari/537.36")

excel_file = r'D:\\1 ir amide\\experiment_carbonyl_origin_group.xlsx'
data = pd.read_excel(excel_file)


In [ ]:
# Configuration
driver_path = 'path_to_chromedriver'  # Replace with your actual path
options = webdriver.ChromeOptions()
options.add_argument('--headless')  # Run in headless mode for efficiency
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')

# Initialize WebDriver
service = Service(driver_path)
driver = webdriver.Chrome(service=service, options=options)

try:
    # Load target website
    driver.get('https://opsin.ch.cam.ac.uk/')
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, 'chemicalName'))
    )

    # Process data
    for i in range(13556, len(data)):
        iupac_name = data['IUPAC_NAME'][i].strip()
        
        if pd.isna(data['SMILES'][i]) and iupac_name:
            try:
                # Input IUPAC name
                input_element = driver.find_element(By.ID, 'chemicalName')
                input_element.clear()
                input_element.send_keys(iupac_name)
                
                # Submit form
                submit_button = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="chemicalNameForm"]/div/button'))
                )
                submit_button.click()
                
                # Wait for response
                WebDriverWait(driver, 10).until(
                    lambda d: d.find_element(By.ID, 'smiles').is_displayed()
                )
                
                # Extract SMILES
                smiles_element = driver.find_element(By.ID, 'smiles')
                data.at[i, 'SMILES'] = smiles_element.text if smiles_element.text else None
                
            except Exception as e:
                print(f"Error processing row {i} ({iupac_name}): {str(e)}")
                data.at[i, 'SMILES'] = None
                # Re-load page if failed
                driver.get('https://opsin.ch.cam.ac.uk/')
                
            # Throttle requests
            time.sleep(1) 

finally:
    driver.quit()
    # Save results
    data.to_excel('D:\\1 ir amide\\output.xlsx', index=False)


C:\Users\13459\AppData\Local\Temp\ipykernel_27364\3327438153.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['IUPAC_NAME'][i] = data['IUPAC_NAME'][i].lstrip().rstrip()


In [ ]:
data.to_excel('D:\\1 ir amide\\output.xlsx', index=False)


In [ ]:
# name to smiles by chemdraw
import pyautogui
import time
import xlrd
import pyperclip
import time
import os
import pandas as pd
import cv2
from tqdm import tqdm

excel_file = r'D:\\1 ir amide\\experiment_carbonyl_no_smiles_group.xlsx'
data = pd.read_excel(excel_file)
img = r'D:\\Jupyter\\auto\\chemdraw.png'
img1 = r'D:\\Jupyter\\auto\\change.png'
img2 = r'D:\\Jupyter\\auto\\false.png'
location=pyautogui.locateCenterOnScreen(img,confidence=0.9)
pyautogui.click(location.x,location.y,clicks=1,interval=0.2,duration=0.2,button='left')

for i in range(0,len(data)):
    if i%10 ==0:
        pyautogui.hotkey('ctrl', 'a')
        pyautogui.hotkey('delete')
    try:
        if (pd.isnull(data['SMILES'][i]) and pd.notnull(data['IUPAC_NAME'][i]) ):
            
            name = data['IUPAC_NAME'][i]
            pyperclip.copy(data['IUPAC_NAME'][i])

            pyautogui.hotkey('ctrl', 'v')  # 按下 Ctrl+V 粘贴
            pyautogui.hotkey('ctrl', 'shift','n')

            pyautogui.hotkey('ctrl', 'alt','c')
            current_content = pyperclip.paste()
            print('匹配：',i,current_content ,'匹配：', name)
            if current_content != name:
                data['SMILES'][i] = current_content
            else:

                time.sleep(4)
                try:
                    location1=pyautogui.locateCenterOnScreen(img1,confidence=0.7)
                    location2=pyautogui.locateCenterOnScreen(img2,confidence=0.7)
                except:
                    pass
                if location1:
                    pyautogui.hotkey('enter')
                    pyautogui.hotkey('ctrl', 'alt','c')
                    current_content = pyperclip.paste()
                    if current_content != name:
                       data['SMILES'][i] = current_content
                elif location2:
                    pyautogui.hotkey('esc')
    finally:
        pyautogui.hotkey('esc')
data.to_excel('D:\\1 ir amide\\output.xlsx', index=False)
